# Section 5.13, Figure 1

Builds a dream-library/router experiment that retrieves or predicts masks for new one-dimensional regression tasks.

## Run Notes
- Run the notebook from top to bottom.
- Synthetic tasks only.
- The final cell builds Figure 1, comparing mask-based methods with random dense baselines.
- Random sampling is intentionally exposed in the experiment cells so readers can vary it.

## Setup

In [ ]:
import copy
import math
import random

import numpy as np
import torch
import torch.autograd as autograd
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt


def signed_kaiming_constant_(
    tensor,
    a=0,
    mode="fan_in",
    nonlinearity="relu",
    k=0.5,
    sparsity=0,
):
    """Initialize frozen weights with signed Kaiming-style uniform samples."""
    fan = nn.init._calculate_correct_fan(tensor, mode)
    gain = nn.init.calculate_gain(nonlinearity, a)
    std = gain / math.sqrt(fan)

    # The active subnetwork keeps a k-fraction of weights, so rescale by 1/sqrt(k).
    if k != 0:
        std *= 1 / math.sqrt(k)

    with torch.no_grad():
        tensor.uniform_(-std, std)
        if sparsity > 0:
            keep_mask = (torch.rand_like(tensor) > sparsity).float()
            tensor *= keep_mask
        return tensor


class GetSubnet(autograd.Function):
    """Straight-through top-k mask used by edge-popup."""

    @staticmethod
    def forward(ctx, scores, k=0.5):
        # Convert scores to a binary mask by keeping the largest k-fraction.
        out = scores.clone()
        _, idx = scores.flatten().sort()
        cutoff = int((1 - k) * scores.numel())
        flat_out = out.flatten()
        flat_out[idx[:cutoff]] = 0
        flat_out[idx[cutoff:]] = 1
        return out

    @staticmethod
    def backward(ctx, grad):
        # Straight-through estimator: pass score gradients through unchanged.
        return grad, None


class LinearSubnet(nn.Linear):
    """Masked linear layer used to build dream-library source models."""

    def __init__(self, in_features, out_features, init=signed_kaiming_constant_, k=0.5, **kwargs):
        super().__init__(in_features, out_features, **kwargs)
        self.k = k
        self.scores = nn.Parameter(torch.randn(out_features, 2 * in_features))
        init(self.weight)
        self.weight.requires_grad_(False)

        if self.bias is not None:
            self.bias_scores = nn.Parameter(torch.randn(2, out_features))
            self.bias.requires_grad_(False)

    def forward(self, x):
        weight_mask = GetSubnet.apply(self.scores.abs(), self.k)
        weight = self.weight * weight_mask[:, : self.weight.shape[-1]]

        if self.bias is None:
            return F.linear(x, weight)

        bias_mask = GetSubnet.apply(self.bias_scores.abs(), self.k)
        bias = self.bias * bias_mask[0, : self.bias.shape[-1]]
        return F.linear(x, weight, bias)


class Network(nn.Module):
    """Small masked MLP used as the source architecture for masks."""

    def __init__(self, layer_sizes, init=signed_kaiming_constant_, bias=True):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            *[
                block
                for layer in layer_sizes
                for block in [LinearSubnet(layer[0], layer[1], init=init, bias=bias), nn.ReLU()]
            ][:-1]
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.layers(x)

Using cuda device


## Experiment and Figure

In [ ]:
# Experiment outline:
# 1. Build a library of synthetic "dream" tasks generated by random binary masks.
# 2. Refit each dream with edge-popup scores and store the resulting hard mask.
# 3. Train a DeepSets router that maps a variable-size probe set to a mask.
# 4. Evaluate retrieval, router prediction, and random dense baselines on sine tasks.

# ============================================================
# Reproducibility / device
# ============================================================

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print("Using device:", device)


# ============================================================
# Actual downstream task family
# ============================================================

def sample_task():
    amp = random.uniform(0.5, 1.5)
    freq = random.uniform(1.0, 4.0)
    phase = random.uniform(-math.pi, math.pi)
    offset = random.uniform(-0.5, 0.5)
    return amp, freq, phase, offset


def eval_task(x, params):
    amp, freq, phase, offset = params
    return amp * torch.sin(freq * x + phase) + offset


def make_dataset(params, n=256, device=device):
    x = torch.linspace(-1.0, 1.0, n, device=device).unsqueeze(1)
    y = eval_task(x[:, 0], params).unsqueeze(1)
    return x, y


# ============================================================
# Probe-set utilities
# ============================================================

def sample_probe_set(num_points,
                     low=-1.0,
                     high=1.0,
                     include_endpoints=True,
                     sort_points=True,
                     device=device):
    if include_endpoints and num_points >= 2:
        interior_n = num_points - 2
        if interior_n > 0:
            interior = low + (high - low) * torch.rand(interior_n, device=device)
            x = torch.cat([
                torch.tensor([low], device=device),
                interior,
                torch.tensor([high], device=device)
            ], dim=0)
        else:
            x = torch.tensor([low], device=device)
            x = torch.cat([x, torch.tensor([high], device=device)], dim=0)
    else:
        x = low + (high - low) * torch.rand(num_points, device=device)

    if sort_points:
        x, _ = torch.sort(x)

    return x.unsqueeze(1)


def sample_probe_count(min_points=16, max_points=256):
    return random.randint(min_points, max_points)


def interp1d_sorted(x_src, y_src, x_query):
    xq = x_query.clamp(min=x_src[0].item(), max=x_src[-1].item())

    idx = torch.searchsorted(x_src, xq)
    idx = torch.clamp(idx, 1, x_src.numel() - 1)

    x0 = x_src[idx - 1]
    x1 = x_src[idx]
    y0 = y_src[idx - 1]
    y1 = y_src[idx]

    denom = (x1 - x0).clamp_min(1e-12)
    t = (xq - x0) / denom
    yq = y0 + t * (y1 - y0)
    return yq


def collate_probe_sets(batch):
    batch_size = len(batch)
    max_len = max(item["pairs"].shape[0] for item in batch)
    mask_dim = batch[0]["mask"].shape[0]

    pairs_padded = torch.zeros(batch_size, max_len, 2, device=device)
    valid_mask = torch.zeros(batch_size, max_len, device=device)
    masks = torch.zeros(batch_size, mask_dim, device=device)

    for i, item in enumerate(batch):
        n = item["pairs"].shape[0]
        pairs_padded[i, :n] = item["pairs"]
        valid_mask[i, :n] = 1.0
        masks[i] = item["mask"]

    return pairs_padded, valid_mask, masks


# ============================================================
# Helpers for actual double-scoring models
# ============================================================

def score_parameters(model):
    params = []
    for layer in model.layers:
        if isinstance(layer, LinearSubnet):
            params.append(layer.scores)
            if layer.bias is not None:
                params.append(layer.bias_scores)
    return params


def reinitialize_scores(model, std=1.0):
    with torch.no_grad():
        for layer in model.layers:
            if isinstance(layer, LinearSubnet):
                layer.scores.normal_(mean=0.0, std=std)
                if layer.bias is not None:
                    layer.bias_scores.normal_(mean=0.0, std=std)


def extract_used_binary_mask_vector(model):
    """
    Extract the actual binary mask used in forward().

    Weight side:
      uses the first half of the doubled score tensor.
    Bias side:
      uses the second row of bias_scores.
    """
    pieces = []

    with torch.no_grad():
        for layer in model.layers:
            if isinstance(layer, LinearSubnet):
                w_mask_full = GetSubnet.apply(layer.scores.abs(), layer.k)
                w_mask_used = w_mask_full[:, :layer.weight.shape[1]]
                pieces.append(w_mask_used.reshape(-1).float())

                if layer.bias is not None:
                    b_mask_full = GetSubnet.apply(layer.bias_scores.abs(), layer.k)
                    b_mask_used = b_mask_full[1, :layer.bias.shape[0]]
                    pieces.append(b_mask_used.reshape(-1).float())

    return torch.cat(pieces, dim=0)


def explicit_forward_with_mask_vector(base_model, x, mask_vec):
    out = x
    offset = 0

    for layer in base_model.layers:
        if isinstance(layer, LinearSubnet):
            w_num = layer.weight.numel()
            w_mask = mask_vec[offset:offset + w_num].view_as(layer.weight)
            offset += w_num

            weight = layer.weight * w_mask

            if layer.bias is not None:
                b_num = layer.bias.numel()
                b_mask = mask_vec[offset:offset + b_num].view_as(layer.bias)
                offset += b_num
                bias = layer.bias * b_mask
            else:
                bias = None

            out = F.linear(out, weight, bias)

        elif isinstance(layer, nn.ReLU):
            out = F.relu(out)

        else:
            raise TypeError(f"Unsupported layer type: {type(layer)}")

    return out


def sample_random_effective_mask_vector(base_model):
    masks = []

    for layer in base_model.layers:
        if not isinstance(layer, LinearSubnet):
            continue

        w_num = layer.weight.numel()
        keep_w = int(round(layer.k * w_num))
        keep_w = max(0, min(keep_w, w_num))

        w_scores = torch.randn(w_num, device=device)
        w_mask = torch.zeros(w_num, device=device)
        if keep_w > 0:
            idx = torch.topk(w_scores, k=keep_w, largest=True, sorted=False).indices
            w_mask[idx] = 1.0
        masks.append(w_mask)

        if layer.bias is not None:
            b_num = layer.bias.numel()
            keep_b = int(round(layer.k * b_num))
            keep_b = max(0, min(keep_b, b_num))

            b_scores = torch.randn(b_num, device=device)
            b_mask = torch.zeros(b_num, device=device)
            if keep_b > 0:
                idx = torch.topk(b_scores, k=keep_b, largest=True, sorted=False).indices
                b_mask[idx] = 1.0
            masks.append(b_mask)

    return torch.cat(masks, dim=0)


def make_base_model(layer_sizes, bias=True, k=0.5):
    model = Network(layer_sizes, bias=bias).to(device)
    for layer in model.layers:
        if isinstance(layer, LinearSubnet):
            layer.k = k
    return model


# ============================================================
# Classical random-network baseline
# ============================================================

class ClassicalMLP(nn.Module):
    def __init__(self, width=512, bias=True):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, width, bias=bias),
            nn.ReLU(),
            nn.Linear(width, width, bias=bias),
            nn.ReLU(),
            nn.Linear(width, 1, bias=bias),
        )

    def forward(self, x):
        return self.net(x)


def evaluate_random_classical_baseline(params,
                                       eval_n=256,
                                       num_random_models=25,
                                       width=512,
                                       bias=True):
    """
    Evaluate a collection of random dense MLPs on the task with NO per-task fitting.
    Returns the empirical mean and spread.
    """
    x_eval, y_eval = make_dataset(params, n=eval_n, device=device)

    eval_mses = []
    with torch.no_grad():
        for _ in range(num_random_models):
            model = ClassicalMLP(width=width, bias=bias).to(device)
            pred = model(x_eval)
            mse = F.mse_loss(pred, y_eval).item()

            eval_mses.append(mse)
    eval_mses = np.array(eval_mses)
    return {
        "eval_mses": eval_mses,
        "mean_eval_mse": float(eval_mses.mean()),
        "best_eval_mse": float(eval_mses.min()),
    }


# ============================================================
# Inner dream fitting with actual double scoring
# ============================================================

def fit_scores_to_probe_behavior(base_model,
                                 x_probe,
                                 y_target_probe,
                                 score_init_std=1.0,
                                 steps=300,
                                 lr=1e-2,
                                 weight_decay=0.0,
                                 verbose=False):
    model = copy.deepcopy(base_model).to(device)
    reinitialize_scores(model, std=score_init_std)

    params = score_parameters(model)
    opt = torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)

    best_state = None
    best_loss = float("inf")
    for step in range(steps):
        opt.zero_grad()
        pred = model(x_probe)
        loss = F.mse_loss(pred.squeeze(1), y_target_probe)
        loss.backward()
        opt.step()

        loss_val = loss.item()
        if loss_val < best_loss:
            best_loss = loss_val
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

        if verbose and (step % 50 == 0 or step == steps - 1):
            print(f"    score-fit step {step:4d} | probe mse = {loss_val:.6f}")

    if best_state is not None:
        model.load_state_dict(best_state)

    with torch.no_grad():
        pred = model(x_probe).squeeze(1)
        hard_mask = extract_used_binary_mask_vector(model)

    return {
        "mask": hard_mask.detach().clone(),
        "probe_mse": F.mse_loss(pred, y_target_probe).item(),
    }


# ============================================================
# Dream library construction:
# random generating mask -> dreamed targets -> double-scoring refit
# ============================================================

def build_dream_library_via_double_scoring(base_model,
                                           num_dreams=1000,
                                           min_probe_n=16,
                                           max_probe_n=256,
                                           score_fit_steps=300,
                                           score_fit_lr=1e-2,
                                           score_init_std=1.0,
                                           low=-1.0,
                                           high=1.0):
    library = []

    print("Building dream library via random-mask dreams + double-scoring refits...")

    for i in range(num_dreams):
        probe_n = sample_probe_count(min_probe_n, max_probe_n)
        x_probe = sample_probe_set(
            num_points=probe_n,
            low=low,
            high=high,
            include_endpoints=True,
            sort_points=True,
            device=device
        )

        source_mask = sample_random_effective_mask_vector(base_model)
        with torch.no_grad():
            y_target_probe = explicit_forward_with_mask_vector(base_model, x_probe, source_mask).squeeze(1)

        fit_result = fit_scores_to_probe_behavior(
            base_model=base_model,
            x_probe=x_probe,
            y_target_probe=y_target_probe,
            score_init_std=score_init_std,
            steps=score_fit_steps,
            lr=score_fit_lr,
            verbose=False
        )

        refit_mask = fit_result["mask"]
        bit_acc = (refit_mask == source_mask).float().mean().item()

        pairs = torch.stack([x_probe[:, 0], y_target_probe], dim=1)

        library.append({
            "x_probe": x_probe[:, 0].detach().clone(),
            "y_target_probe": y_target_probe.detach().clone(),
            "pairs": pairs.detach().clone(),
            "mask": refit_mask.detach().clone(),
            "fit_mse": fit_result["probe_mse"],
            "source_vs_refit_bit_acc": bit_acc,
        })

        if (i + 1) % 50 == 0 or i == num_dreams - 1:
            print(
                f"  built {i+1}/{num_dreams} | "
                f"latest fit mse = {fit_result['probe_mse']:.6f} | "
                f"source/refit bit acc = {bit_acc:.4f}"
            )

    return library


# ============================================================
# Retrieval baseline
# ============================================================

def retrieve_best_dream_mask(query_x_probe,
                             query_y_probe,
                             dream_library):
    if query_x_probe.ndim == 2:
        xq = query_x_probe[:, 0]
    else:
        xq = query_x_probe

    if query_y_probe.ndim == 2:
        yq = query_y_probe[:, 0]
    else:
        yq = query_y_probe

    mses = []

    with torch.no_grad():
        for item in dream_library:
            xd = item["x_probe"]
            yd = item["y_target_probe"]
            yd_interp = interp1d_sorted(xd, yd, xq)
            mse = torch.mean((yd_interp - yq) ** 2)

            mses.append(mse)

        mses = torch.stack(mses, dim=0)
        best_idx = torch.argmin(mses).item()

    best_item = dream_library[best_idx]
    return {
        "best_mask": best_item["mask"].detach().clone(),
        "best_probe_mse": mses[best_idx].item(),
        "fit_mse_of_selected_dream": best_item["fit_mse"],
        "source_vs_refit_bit_acc_of_selected_dream": best_item["source_vs_refit_bit_acc"],
    }


# ============================================================
# Set-based router (DeepSets)
# ============================================================

class DeepSetRouter(nn.Module):
    def __init__(self, mask_dim, elem_hidden=256, set_hidden=512):
        super().__init__()

        self.phi = nn.Sequential(
            nn.Linear(2, elem_hidden),
            nn.ReLU(),
            nn.Linear(elem_hidden, elem_hidden),
            nn.ReLU(),
        )

        self.rho = nn.Sequential(
            nn.Linear(elem_hidden, set_hidden),
            nn.ReLU(),
            nn.Linear(set_hidden, set_hidden),
            nn.ReLU(),
            nn.Linear(set_hidden, mask_dim),
        )

    def forward(self, pairs, valid_mask):
        elem_embed = self.phi(pairs)
        vm = valid_mask.unsqueeze(-1)
        elem_embed = elem_embed * vm

        denom = vm.sum(dim=1).clamp_min(1.0)
        pooled = elem_embed.sum(dim=1) / denom

        logits = self.rho(pooled)
        return logits


def train_router_on_dreams(dream_library,
                           mask_dim,
                           elem_hidden=256,
                           set_hidden=512,
                           epochs=1000,
                           lr=1e-3,
                           batch_size=64):
    router = DeepSetRouter(
        mask_dim=mask_dim,
        elem_hidden=elem_hidden,
        set_hidden=set_hidden
    ).to(device)

    opt = torch.optim.Adam(router.parameters(), lr=lr)

    n = len(dream_library)

    for epoch in range(epochs):
        perm = torch.randperm(n, device=device)
        epoch_loss = 0.0

        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size].tolist()
            batch = [dream_library[i] for i in idx]

            pairs_padded, valid_mask, target_masks = collate_probe_sets(batch)

            opt.zero_grad()
            logits = router(pairs_padded, valid_mask)
            loss = F.binary_cross_entropy_with_logits(logits, target_masks)
            loss.backward()
            opt.step()

            epoch_loss += loss.item() * len(batch)

        epoch_loss /= n

        if epoch % 100 == 0 or epoch == epochs - 1:
            with torch.no_grad():
                total_correct = 0.0
                total_bits = 0.0

                for start in range(0, n, batch_size):
                    batch = dream_library[start:start + batch_size]
                    pairs_padded, valid_mask, target_masks = collate_probe_sets(batch)

                    logits = router(pairs_padded, valid_mask)
                    probs = torch.sigmoid(logits)
                    hard = (probs > 0.5).float()

                    total_correct += (hard == target_masks).float().sum().item()
                    total_bits += target_masks.numel()

                bit_acc = total_correct / total_bits

            print(f"Router epoch={epoch:4d} | loss={epoch_loss:.6f} | bit_acc={bit_acc:.4f}")

    return router


def router_predict_mask(router, query_x_probe, query_y_probe, hard=True, threshold=0.5):
    with torch.no_grad():
        if query_x_probe.ndim == 2:
            x = query_x_probe[:, 0]
        else:
            x = query_x_probe

        if query_y_probe.ndim == 2:
            y = query_y_probe[:, 0]
        else:
            y = query_y_probe

        pairs = torch.stack([x, y], dim=1).unsqueeze(0)
        valid_mask = torch.ones(1, pairs.shape[1], device=device)

        logits = router(pairs, valid_mask).squeeze(0)
        probs = torch.sigmoid(logits)

        if hard:
            mask = (probs > threshold).float()
        else:
            mask = probs

    return mask


# ============================================================
# Evaluation
# ============================================================

def evaluate_router_and_retrieval_on_new_task(base_model,
                                              dream_library,
                                              router,
                                              params,
                                              min_probe_n=16,
                                              max_probe_n=256,
                                              eval_n=256,
                                              router_hard=True,
                                              num_random_classical=25,
                                              classical_width=512,
                                              classical_bias=True):
    probe_n = sample_probe_count(min_probe_n, max_probe_n)
    x_probe = sample_probe_set(
        num_points=probe_n,
        low=-1.0,
        high=1.0,
        include_endpoints=True,
        sort_points=True,
        device=device
    )
    y_probe = eval_task(x_probe[:, 0], params).unsqueeze(1)

    x_eval, y_eval = make_dataset(params, n=eval_n, device=device)

    retrieval = retrieve_best_dream_mask(
        query_x_probe=x_probe,
        query_y_probe=y_probe,
        dream_library=dream_library
    )

    router_mask = router_predict_mask(
        router=router,
        query_x_probe=x_probe,
        query_y_probe=y_probe,
        hard=router_hard,
        threshold=0.5
    )

    classical = evaluate_random_classical_baseline(
        params=params,
        eval_n=eval_n,
        num_random_models=num_random_classical,
        width=classical_width,
        bias=classical_bias
    )

    with torch.no_grad():
        pred_eval_retrieval = explicit_forward_with_mask_vector(base_model, x_eval, retrieval["best_mask"])
        pred_eval_router = explicit_forward_with_mask_vector(base_model, x_eval, router_mask)

        eval_mse_retrieval = F.mse_loss(pred_eval_retrieval, y_eval).item()
        eval_mse_router = F.mse_loss(pred_eval_router, y_eval).item()

        probe_pred_router = explicit_forward_with_mask_vector(base_model, x_probe, router_mask).squeeze(1)
        best_probe_mse_router = F.mse_loss(probe_pred_router, y_probe.squeeze(1)).item()

    return {
        "selected_dream_fit_mse": retrieval["fit_mse_of_selected_dream"],
        "selected_dream_source_vs_refit_bit_acc": retrieval["source_vs_refit_bit_acc_of_selected_dream"],
        "best_probe_mse_retrieval": retrieval["best_probe_mse"],
        "best_probe_mse_router": best_probe_mse_router,
        "classical_eval_mses": classical["eval_mses"],
        "classical_mean_eval_mse": classical["mean_eval_mse"],
        "classical_best_eval_mse": classical["best_eval_mse"],
        "eval_mse_retrieval": eval_mse_retrieval,
        "eval_mse_router": eval_mse_router,
    }


# ============================================================
# Final figure
# ============================================================


def plot_mask_vs_classical_boxplots(results, max_tasks_to_show=30, sort_by="classical_median"):
    """
    For each task, show the distribution of random classical-network eval MSEs
    as a boxplot, and overlay retrieval/router eval MSEs.

    Interpretation:
      - retrieval/router point below the box => better than most random classical draws
      - inside the box => comparable to random classical distribution
      - above the box => worse than most random classical draws
    """
    n = min(len(results), max_tasks_to_show)

    if sort_by == "classical_median":
        order = np.argsort([np.median(r["classical_eval_mses"]) for r in results])[:n]
    elif sort_by == "classical_mean":
        order = np.argsort([np.mean(r["classical_eval_mses"]) for r in results])[:n]
    elif sort_by == "retrieval":
        order = np.argsort([r["eval_mse_retrieval"] for r in results])[:n]
    elif sort_by == "router":
        order = np.argsort([r["eval_mse_router"] for r in results])[:n]
    else:
        order = np.arange(n)

    chosen = [results[i] for i in order]

    classical_distributions = [r["classical_eval_mses"] for r in chosen]
    retrieval_vals = [r["eval_mse_retrieval"] for r in chosen]
    router_vals = [r["eval_mse_router"] for r in chosen]

    positions = np.arange(1, n + 1)

    plt.figure(figsize=(max(12, 0.45 * n), 6))

    plt.boxplot(
        classical_distributions,
        positions=positions,
        widths=0.6,
        patch_artist=False,
        showfliers=False,
        whis=(5, 95),
    )

    plt.scatter(positions, retrieval_vals, label="Retrieval", zorder=3, s=30)
    plt.scatter(positions, router_vals, label="Router", zorder=3, s=30)

    plt.xlabel("Task index")
    plt.ylabel("Eval MSE")
    plt.title("Mask-based methods vs random classical distribution per task")
    plt.yscale("log")
    plt.grid(True, axis="y", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

# ============================================================
# Run experiment
# ============================================================

# Base network: frozen weights, trainable scores only when cloned per dream
layer_sizes = [[1, 128], [128, 128], [128, 1]]
bias = True
k = 0.5

base_model = make_base_model(layer_sizes=layer_sizes, bias=bias, k=k)
mask_dim = extract_used_binary_mask_vector(base_model).numel()
print("Mask dimension:", mask_dim)

# Dream library construction
num_dreams = 1000
min_probe_n = 16
max_probe_n = 256

score_fit_steps = 500
score_fit_lr = 1e-2
score_init_std = 1.0

dream_library = build_dream_library_via_double_scoring(
    base_model=base_model,
    num_dreams=num_dreams,
    min_probe_n=min_probe_n,
    max_probe_n=max_probe_n,
    score_fit_steps=score_fit_steps,
    score_fit_lr=score_fit_lr,
    score_init_std=score_init_std,
    low=-1.0,
    high=1.0
)

print("Dream library size:", len(dream_library))

# Router
router_elem_hidden = 256
router_set_hidden = 1024
router_epochs = 1000
router_lr = 1e-3
router_batch_size = 64

router = train_router_on_dreams(
    dream_library=dream_library,
    mask_dim=mask_dim,
    elem_hidden=router_elem_hidden,
    set_hidden=router_set_hidden,
    epochs=router_epochs,
    lr=router_lr,
    batch_size=router_batch_size
)

# Evaluation on actual sine tasks
num_test_tasks = 80
eval_n = 256

num_random_classical = 25
classical_width = 512
classical_bias = True

results = []
for i in range(num_test_tasks):
    params = sample_task()
    r = evaluate_router_and_retrieval_on_new_task(
        base_model=base_model,
        dream_library=dream_library,
        router=router,
        params=params,
        min_probe_n=min_probe_n,
        max_probe_n=max_probe_n,
        eval_n=eval_n,
        router_hard=True,
        num_random_classical=num_random_classical,
        classical_width=classical_width,
        classical_bias=classical_bias
    )
    results.append(r)

    if (i + 1) % 20 == 0 or i == num_test_tasks - 1:
        print(f"Evaluated {i+1}/{num_test_tasks} tasks")

eval_mses_retrieval = [r["eval_mse_retrieval"] for r in results]
eval_mses_router = [r["eval_mse_router"] for r in results]
probe_mses_retrieval = [r["best_probe_mse_retrieval"] for r in results]
probe_mses_router = [r["best_probe_mse_router"] for r in results]
selected_fit_mses = [r["selected_dream_fit_mse"] for r in results]
selected_source_vs_refit = [r["selected_dream_source_vs_refit_bit_acc"] for r in results]
classical_mean_evals = [r["classical_mean_eval_mse"] for r in results]
classical_best_evals = [r["classical_best_eval_mse"] for r in results]

print("\nRandom-mask-dream + double-scoring-refit results")
print("------------------------------------------------")
print("Mean retrieval eval MSE:          ", float(np.mean(eval_mses_retrieval)))
print("Mean router eval MSE:             ", float(np.mean(eval_mses_router)))
print("Mean random classical mean MSE:   ", float(np.mean(classical_mean_evals)))
print("Median retrieval eval MSE:        ", float(np.median(eval_mses_retrieval)))
print("Median router eval MSE:           ", float(np.median(eval_mses_router)))
print("Median random classical mean MSE: ", float(np.median(classical_mean_evals)))
print("Mean retrieval probe MSE:         ", float(np.mean(probe_mses_retrieval)))
print("Mean router probe MSE:            ", float(np.mean(probe_mses_router)))
print("Mean selected dream fit MSE:      ", float(np.mean(selected_fit_mses)))
print("Mean selected source/refit acc:   ", float(np.mean(selected_source_vs_refit)))
print("Best retrieval eval MSE:          ", float(np.min(eval_mses_retrieval)))
print("Best router eval MSE:             ", float(np.min(eval_mses_router)))
print("Best random classical mean MSE:   ", float(np.min(classical_mean_evals)))
print("Mean random classical best-of-draw MSE:", float(np.mean(classical_best_evals)))
print("Worst retrieval eval MSE:         ", float(np.max(eval_mses_retrieval)))
print("Worst router eval MSE:            ", float(np.max(eval_mses_router)))
print("Worst random classical mean MSE:  ", float(np.max(classical_mean_evals)))

# Figure 1: compare mask-based methods against the random classical spread.
plot_mask_vs_classical_boxplots(
    results,
    max_tasks_to_show=30,
    sort_by="classical_median"
)

Using device: cuda
Mask dimension: 16897
Building dream library via random-mask dreams + double-scoring refits...
  built 50/1000 | latest fit mse = 0.000014 | source/refit bit acc = 0.4938
  built 100/1000 | latest fit mse = 0.000017 | source/refit bit acc = 0.5033
  built 150/1000 | latest fit mse = 0.000006 | source/refit bit acc = 0.5018
  built 200/1000 | latest fit mse = 0.000012 | source/refit bit acc = 0.4986
  built 250/1000 | latest fit mse = 0.000010 | source/refit bit acc = 0.5010
  built 300/1000 | latest fit mse = 0.000007 | source/refit bit acc = 0.5006
  built 350/1000 | latest fit mse = 0.000021 | source/refit bit acc = 0.4969
  built 400/1000 | latest fit mse = 0.000219 | source/refit bit acc = 0.4993
  built 450/1000 | latest fit mse = 0.000016 | source/refit bit acc = 0.5126
  built 500/1000 | latest fit mse = 0.000236 | source/refit bit acc = 0.5043
  built 550/1000 | latest fit mse = 0.000017 | source/refit bit acc = 0.4939
